# Breaking the Enigma with permutation tensors

*others · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/enigma_solver)

Not physics, just for fun: every key press of a 3-rotor Enigma I is a product of eight $26\times26$ permutation matrices, with each rotor conjugated by a cyclic shift for its position. Encrypt an English message with a secret rotor order and start position, then break it by decrypting with all $26^3 = 17\,576$ start positions at once as one batch tensor, for every rotor order, and keeping the text whose letter statistics look most like English.

**Vector form**

$$E = S^{-1} R_1^{-1} R_2^{-1} R_3^{-1}\, U\, R_3 R_2 R_1\, S,\qquad R_k = \rho^{-o_k}\, W_k\, \rho^{o_k},\qquad E^2 = I,\quad E_{ii} = 0$$

**Index (tensor) form**

$$R^{(k)}_{ij} = \rho^{-o_k}_{ia} W^{(k)}_{ab}\, \rho^{o_k}_{bj} \qquad E_{ij} = S_{ai} R^{(1)}_{ba} R^{(2)}_{cb} R^{(3)}_{dc}\, U_{de}\, R^{(3)}_{ef} R^{(2)}_{fg} R^{(1)}_{gh} S_{hj} \qquad c_{bk} = \textstyle\sum_t [y_{bt} = k] \qquad \mathrm{IoC}_b = \frac{c_{bk}(c_{bk} - 1)}{N(N - 1)}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $e_x$ | `to_index(text)` | (T,) | a letter is a one-hot vector $e_x$ or just its index $x \in \{0, \dots, 25\}$ (A = 0) |
| $W_{ab}$ | `matrix(perm(ROTORS[name][0]))` | (26, 26) | historical wiring of rotor I–V: $W_{ab} = 1$ when contact $b$ is wired to contact $a$ |
| $\rho^{o}_{ij}$ | `shift(o)` | (T, 26, 26) | cyclic shift $x \to x + o$; $o_k$ = window letter − ring setting of rotor $k$ at key press $t$ |
| $R^{(k)}_{ij}$ | `einsum('tia,ab,tbj->tij', ...)` | (T, 26, 26) | repeated $a, b$ ⇒ summed: the turned rotor; as indices $x \to (w[(x + o) \bmod 26] - o) \bmod 26$ |
| $E_{ij}$ | `einsum('ai,tba,tcb,tdc,de,tef,tfg,tgh,hj->tij', ...)` | (T, 26, 26) | one key press: plugboard $S$, rotors right $R^{(1)}$ to left $R^{(3)}$, reflector $U$ and back; $(R^{-1})_{ab} = R_{ba}$ |
| $F_{ox}$ | `rotor_table(w)` | (26, 26) | the rotor at all 26 offsets as a look-up table, so $R\,e_x$ becomes F[o, x] for millions of letters at once |
| $y_{bt}$ | `encipher(text, offsets, m, s)` | (B, T) | the ciphertext decrypted with every start position $b$ ($B = 26^3$); decryption = encryption since $E^2 = I$ |
| $c_{bk}$ | `counts.scatter_add_(1, y, ones)` | (B, 26) | how often letter $k$ occurs in candidate $b$ |
| $\mathrm{IoC}_b$ | `einsum('bk,bk->b', c, c - 1) / (N*(N-1))` | (B,) | repeated $k$ ⇒ summed: chance that two letters agree, about 0.066 for English and $1/26 \approx 0.038$ for random text |
| $L_b$ | `einsum('bk,k->b', c, log_p) / N` | (B,) | the alternative score: mean English unigram log-likelihood $c_{bk}\log p_k / N$ |

The reflector $U$ is a symmetric permutation without fixed points, so $E = Q^{-1}UQ$ with $Q = R_3R_2R_1S$ is one too: $E^2 = I$ (the same setting decrypts) and $E_{ii} = 0$ (no letter is ever encrypted to itself, a weakness the codebreakers used). Before each key press the right rotor steps; when it sits at its notch it also kicks the middle rotor, and when the middle rotor sits at its own notch it steps together with the left rotor (the double-step anomaly). Test vector: rotors I-II-III, reflector B, rings AAA, start AAA turns AAAAA into BDZGO. History: Marian Rejewski broke the Enigma in 1932 with permutation group theory, and Turing and Welchman's bombe (1940) searched rotor orders and positions using cribs, guessed plaintext such as weather reports, not statistics. The purely statistical attack here, ranking decryptions by the index of coincidence, is the one James Gillogly published in 1995 (Cryptologia), which also recovered ring settings and plugs in later stages. Simplifications: the ring settings are AAA and known to the attacker, and the plugboard is empty, so the key space is only $60 \times 26^3 \approx 10^6$ settings and the correct one stands out clearly. A real wartime plugboard with 10 cables adds about $1.5 \times 10^{14}$ choices, swaps 20 of the 26 letters on both sides of the rotors, and scrambles the letter statistics so much that the correct rotor setting barely beats the wrong ones; Gillogly's attack then needs a hill climb over the plugs, and with short messages it fails. Try fewer letters (about 60 to 100) or the other score to see when the statistics stop being reliable; searching only rotors I–III cannot find a key that uses IV or V.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/enigma_solver/main.py
"""Main loop: parameters → tensors → encrypt → break → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors

RINGS = "AAA"  # ring settings of the secret key; the attacker is assumed to know them


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    rings, plugs = tensors.to_index(RINGS, dev), tensors.plugboard("", dev)  # empty plugboard S = I
    order = tuple(params["rotor_order"].split("-"))
    start = tensors.to_index(params["start_left"] + params["start_middle"] + params["start_right"], dev)  # (3,)
    m = tensors.machine(order, dev)

    # 1. The machine: a known test vector, then the secret message in matrix form and in index form.
    test = solve.run_machine(tensors.to_index("AAAAA", dev), tensors.machine(("I", "II", "III"), dev),
                             tensors.to_index("AAA", dev)[None], rings, plugs)  # fmt: skip
    print(f"test vector: rotors I-II-III, reflector B, rings AAA, start AAA: AAAAA → {tensors.to_text(test[0])}")
    plain = tensors.to_index(tensors.MESSAGES[params["message"]], dev)  # (T,)
    e = solve.keypress_matrices(m, start, rings, plugs, len(plain))  # (T, 26, 26), one E_t per key press
    eye = torch.eye(26, device=dev)
    print(f"secret key: rotors {params['rotor_order']}, rings {RINGS}, start {tensors.to_text(start)}, no plugs")
    print(f"{len(plain)} key presses: E_t E_t = I for all t: {torch.equal(e @ e, eye.expand_as(e))};"
          f"  letters mapped to themselves: {int(e.diagonal(dim1=-2, dim2=-1).sum())}")  # fmt: skip
    cipher_matrix = torch.einsum("tij,tj->ti", e, tensors.matrix(plain).transpose(-1, -2)).argmax(-1)  # E_t e_{x_t}
    cipher = solve.run_machine(plain, m, start[None], rings, plugs)[0]  # (T,) the same with table look-ups
    pos = solve.positions(start[None], m["notch"], len(plain))[0]  # (T, 3) window letters at each key press
    before = torch.cat([start[None], pos[:-1]])  # (T, 3) window letters just before each key press
    double_steps = int((pos[:, 0] != before[:, 0]).sum())  # each left step comes with a middle double step
    print(f"matrix form = index form: {torch.equal(cipher_matrix, cipher)};  double steps: {double_steps}")
    groups = " ".join(tensors.to_text(cipher[i : i + 5]) for i in range(0, 60, 5))
    print(f"plaintext : {tensors.to_text(plain[:60])}…\nciphertext: {groups} …")

    # 2. The attack: decrypt with all 26³ start positions for every rotor order and keep the most English-looking.
    orders = tensors.rotor_orders(params["search_rotors"])
    n = min(params["n_letters"], len(cipher), 120 if fast_mode() else len(cipher))
    tic = time.perf_counter()
    result = solve.break_enigma(cipher[:n], orders, rings, params["score"])
    seconds = time.perf_counter() - tic
    scores = result["scores"]  # (O, 26, 26, 26)
    best_per_order = scores.flatten(1).amax(1)  # (O,)
    print(f"scored {len(orders)} rotor orders × 26³ = {len(orders) * 26**3:,} keys on {n} letters "
          f"({params['score']}) in {seconds:.1f} s")  # fmt: skip
    for i in best_per_order.argsort(descending=True)[:4].tolist():
        print(f"  {'-'.join(orders[i]):>11}: best score {best_per_order[i].item():.4f}")
    best_order, best_start = orders[result["order"]], result["start"]
    found = best_order == order and torch.equal(best_start, start)
    print(f"recovered: rotors {'-'.join(best_order)}, start {tensors.to_text(best_start)}  "
          f"({'correct' if found else 'wrong: try more letters or the other score'})")  # fmt: skip

    # 3. Decrypt the whole message with the recovered key (decryption = encryption), and a preview per order.
    decrypted = solve.run_machine(cipher, tensors.machine(best_order, dev), best_start[None], rings, plugs)[0]
    print(f"decrypted : {tensors.to_text(decrypted)}")
    previews = []
    for i, o in enumerate(orders):
        b = int(scores[i].flatten().argmax())
        s_i = tensors.all_starts(dev)[b]
        y = solve.run_machine(cipher[:40], tensors.machine(o, dev), s_i[None], rings, plugs)[0]
        previews.append(f"{'-'.join(o)} at {tensors.to_text(s_i)}: {tensors.to_text(y)}…")
    return plot.animate(
        scores.amax(1), best_per_order, orders, previews, tensors.to_text(cipher), tensors.to_text(decrypted),
        result["order"], best_start, params["score"], out_dir / "enigma_solver.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/enigma_solver/equation.py
r"""Machine → tensor translation.

The Enigma is not physics, but it is a beautiful product of permutation matrices. A letter x is a one-hot vector
e_x (or just its index). A wiring W is a 26×26 permutation matrix, W e_x = e_{w[x]}. A rotor turned o steps away
from its rest position is the same wiring seen through a cyclic shift ρ (ρ e_x = e_{x+1}):

    R = ρ^{-o} W ρ^{o}        index form:  R_{ij} = (ρ^{-o})_{ia} W_{ab} (ρ^{o})_{bj}     (a, b summed)
                              as indices:  x → (w[(x + o) mod 26] − o) mod 26

One key press sends the current through the plugboard S, the fast right rotor R₁, the middle R₂, the slow left R₃,
the reflector U and back again:

    E = S⁻¹ R₁⁻¹ R₂⁻¹ R₃⁻¹ U R₃ R₂ R₁ S
    E_{ij} = S_{ai} R₁_{ba} R₂_{cb} R₃_{dc} U_{de} R₃_{ef} R₂_{fg} R₁_{gh} S_{hj}     (a … h summed)

using R⁻¹ = Rᵀ for a permutation matrix. U is a symmetric involution without fixed points (U² = I, U_ii = 0), so
E = Q⁻¹ U Q is too: E² = I (encryption = decryption) and E_ii = 0 (no letter is ever encrypted to itself).
Before each key press the rotors step: the right rotor always; the middle one when the right rotor sits at its
notch; the left one, and the middle one again, when the middle rotor sits at its own notch (the double step).
"""

import torch


def rotor_matrix(w: torch.Tensor, rho_minus: torch.Tensor, rho_plus: torch.Tensor) -> torch.Tensor:
    r"""R_{tij} = (ρ^{-o_t})_{ia} W_{ab} (ρ^{o_t})_{bj}   (a, b summed).   W (26, 26), ρ (T, 26, 26) → (T, 26, 26)"""
    return torch.einsum("tia,ab,tbj->tij", rho_minus, w, rho_plus)


def keypress_matrix(s: torch.Tensor, r1: torch.Tensor, r2: torch.Tensor, r3: torch.Tensor, u: torch.Tensor):
    r"""E_{tij} = S_{ai} R1_{tba} R2_{tcb} R3_{tdc} U_{de} R3_{tef} R2_{tfg} R1_{tgh} S_{hj}   (a … h summed).

    S, U (26, 26); R1 (right), R2 (middle), R3 (left) (T, 26, 26) → (T, 26, 26), one matrix per key press.
    """
    return torch.einsum("ai,tba,tcb,tdc,de,tef,tfg,tgh,hj->tij", s, r1, r2, r3, u, r3, r2, r1, s)


def rotor(x: torch.Tensor, w: torch.Tensor, o: torch.Tensor) -> torch.Tensor:
    r"""Index form of R = ρ^{-o} W ρ^{o}: x → (w[(x + o) mod 26] − o) mod 26.   Any shape, x and o broadcast."""
    return (w[(x + o) % 26] - o) % 26


def rotor_table(w: torch.Tensor) -> torch.Tensor:
    r"""F_{ox} = (w[(x + o) mod 26] − o) mod 26: the rotor at every offset o (row o is R for that o).

    (26,) → (26, 26)
    """
    a = torch.arange(26, device=w.device)
    return rotor(a[None, :], w, a[:, None])


def encipher(x: torch.Tensor, offsets: torch.Tensor, m: dict, s: torch.Tensor) -> torch.Tensor:
    r"""y = S⁻¹ R₁⁻¹ R₂⁻¹ R₃⁻¹ U R₃ R₂ R₁ S x for a whole batch of letters.

    x (...,) letter indices; offsets (..., 3) = window position − ring setting of (left, mid, right);
    m holds the rotor tables f, f_inv (3, 26, 26) (see rotor_table) and the reflector u (26,); s is the plugboard.
    Each rotor is one table look-up F[o, x], which is the same as R x but fast for millions of letters.
    """
    y = s[x]
    for k in (2, 1, 0):  # right → middle → left
        y = m["f"][k][offsets[..., k], y]
    y = m["u"][y]
    for k in (0, 1, 2):  # left → middle → right, through the inverse wirings
        y = m["f_inv"][k][offsets[..., k], y]
    return s[y]  # S is its own inverse


def step(pos: torch.Tensor, notch: torch.Tensor) -> torch.Tensor:
    """One key press of the stepping mechanism, double step included.   pos (..., 3) window letters → (..., 3)"""
    mid_at_notch = pos[..., 1] == notch[1]
    right_at_notch = pos[..., 2] == notch[2]
    kick = torch.stack([mid_at_notch, mid_at_notch | right_at_notch, torch.ones_like(mid_at_notch)], dim=-1)
    return (pos + kick.long()) % 26


def letter_counts(y: torch.Tensor) -> torch.Tensor:
    r"""c_{bk} = \sum_t [y_{bt} = k]: how often each letter occurs in every candidate text.   (B, T) → (B, 26)"""
    counts = torch.zeros(y.shape[0], 26, device=y.device)
    return counts.scatter_add_(1, y, torch.ones_like(y, dtype=counts.dtype))


def index_of_coincidence(c: torch.Tensor) -> torch.Tensor:
    r"""IoC_b = c_{bk}(c_{bk} − 1) / (N(N − 1))   (k summed): English ≈ 0.066, random ≈ 0.038.   (B, 26) → (B,)"""
    n = c[0].sum()
    return torch.einsum("bk,bk->b", c, c - 1) / (n * (n - 1))


def unigram_score(c: torch.Tensor, log_p: torch.Tensor) -> torch.Tensor:
    r"""L_b = c_{bk} log p_k / N   (k summed): mean log-likelihood per letter under English.   (B, 26) → (B,)"""
    return torch.einsum("bk,k->b", c, log_p) / c[0].sum()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/enigma_solver/tensors.py
"""Build the concrete tensors: rotor wirings, the reflector, the plugboard, the messages and every start position.

Letters are indices 0–25 (A = 0). A wiring is stored as a permutation vector w of shape (26,): contact x is wired
to contact w[x]. The same permutation as a 26×26 matrix has M[w[x], x] = 1, so M @ e_x = e_{w[x]} for a one-hot e_x.
"""

import itertools
import string

import torch

from .equation import rotor_table

ALPHABET = string.ascii_uppercase

# Historical Enigma I wirings (entry contact A..Z → exit contact) and the window letter at which each rotor
# kicks its left neighbour on the next key press (the notch).
ROTORS = {
    "I": ("EKMFLGDQVZNTOWYHXUSPAIBRCJ", "Q"),
    "II": ("AJDKSIRUXBLHWTMCQGZNPYFVOE", "E"),
    "III": ("BDFHJLCPRTXVZNYEIWGAKMUSQO", "V"),
    "IV": ("ESOVPZJAYQUIRHXLNFTGKDCMWB", "J"),
    "V": ("VZBRGITYUPSDNHLXAWMJQOFECK", "Z"),
}
REFLECTOR_B = "YRUHQSLDPXNGOKMIEBFZCWVJAT"

# English letter frequencies in percent, A..Z (standard tables from large text corpora).
ENGLISH_FREQ = [
    8.167, 1.492, 2.782, 4.253, 12.702, 2.228, 2.015, 6.094, 6.966, 0.153, 0.772, 4.025, 2.406,
    6.749, 7.507, 1.929, 0.095, 5.987, 6.327, 9.056, 2.758, 0.978, 2.360, 0.150, 1.974, 0.074,
]  # fmt: skip

# Self-written plaintexts (no copyright); spaces and punctuation are dropped before encryption.
MESSAGES = {
    "weather": (
        "Weather report for the northern sector at six hundred hours. Visibility is good along the coast with "
        "light winds from the west and a calm sea. Clouds will build over the hills during the afternoon and rain "
        "is expected after dark. Supply ships should leave the harbour before noon and keep close to the shore "
        "until the storm has passed. Report any change in the barometer to headquarters at once."
    ),
    "lighthouse": (
        "The old lighthouse keeper climbed the spiral stairs every evening to light the lamp. From the top he "
        "could see the fishing boats returning with the tide and the gulls circling above the rocks. In winter "
        "the waves were so high that spray reached the windows of the lantern room, yet the light never failed. "
        "Sailors said that as long as the beam swept across the bay, nobody would be lost at sea."
    ),
    "permutations": (
        "A permutation is a square matrix with exactly one entry equal to one in every row and every column. "
        "Multiplying such matrices composes the permutations, and the transpose undoes them. The Enigma machine "
        "multiplied eight of these matrices for every key that was pressed, and because the rotors moved after "
        "each letter the product changed all the time. Breaking it means searching over the starting positions "
        "until the plaintext looks like English again."
    ),
}


def to_index(text: str, device: torch.device | str = "cpu") -> torch.Tensor:
    """Keep only the letters of text and turn them into indices: (T,) long."""
    return torch.tensor([ALPHABET.index(c) for c in text.upper() if c in ALPHABET], dtype=torch.long, device=device)


def to_text(x: torch.Tensor) -> str:
    """Indices (T,) → string."""
    return "".join(ALPHABET[i] for i in x.tolist())


def perm(letters: str, device: torch.device | str = "cpu") -> torch.Tensor:
    """A wiring string such as 'EKMF…' as a permutation vector w: (26,) long."""
    return to_index(letters, device)


def inverse(w: torch.Tensor) -> torch.Tensor:
    """w⁻¹ with w⁻¹[w[x]] = x: (26,) → (26,)."""
    inv = torch.empty_like(w)
    inv[w] = torch.arange(26, device=w.device)
    return inv


def matrix(w: torch.Tensor) -> torch.Tensor:
    """Permutation matrix M_{ij} = [i = w[j]], so that M e_x = e_{w[x]}: (..., 26) → (..., 26, 26)."""
    return torch.nn.functional.one_hot(w, 26).transpose(-1, -2).float()


def shift(o: torch.Tensor) -> torch.Tensor:
    """Cyclic shift ρ^o as a matrix (x → x + o): o (...,) → (..., 26, 26)."""
    x = torch.arange(26, device=o.device)
    return matrix((x + o[..., None]) % 26)


def plugboard(pairs: str, device: torch.device | str = "cpu") -> torch.Tensor:
    """Steckerbrett S as a permutation vector: 'AB CD' swaps A↔B and C↔D, every other letter is unplugged: (26,)."""
    s = torch.arange(26, device=device)
    for a, b in pairs.upper().split():
        i, j = ALPHABET.index(a), ALPHABET.index(b)
        s[i], s[j] = j, i
    return s


def machine(order: tuple[str, ...], device: torch.device | str = "cpu") -> dict[str, torch.Tensor]:
    """Wirings W, W⁻¹ (3, 26), their tables (3, 26, 26), notches (3,) and reflector U (26,) for (left, mid, right)."""
    w = torch.stack([perm(ROTORS[name][0], device) for name in order])  # (3, 26)
    w_inv = torch.stack([inverse(r) for r in w])  # (3, 26)
    return {
        "w": w,
        "w_inv": w_inv,
        "f": torch.stack([rotor_table(r) for r in w]),  # (3, 26, 26): F[k, o, x]
        "f_inv": torch.stack([rotor_table(r) for r in w_inv]),
        "notch": to_index("".join(ROTORS[name][1] for name in order), device),
        "u": perm(REFLECTOR_B, device),
    }


def rotor_orders(rotor_set: str) -> list[tuple[str, str, str]]:
    """Every ordered choice of 3 different rotors: 6 orders from I–III, 60 from I–V."""
    names = ["I", "II", "III"] if rotor_set == "I-III" else list(ROTORS)
    return list(itertools.permutations(names, 3))


def all_starts(device: torch.device | str = "cpu") -> torch.Tensor:
    """All 26³ = 17,576 window settings (left, mid, right), row b = 676·left + 26·mid + right: (B, 3)."""
    a = torch.arange(26, device=device)
    return torch.cartesian_prod(a, a, a)


def english_log_probs(device: torch.device | str = "cpu") -> torch.Tensor:
    """log p_k of the English unigram distribution: (26,)."""
    p = torch.tensor(ENGLISH_FREQ, device=device)
    return torch.log(p / p.sum())


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/enigma_solver/solve.py
"""Run the machine for a batch of settings, and break a message by scoring every rotor order and start position."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import encipher, index_of_coincidence, keypress_matrix, letter_counts, rotor_matrix, step, unigram_score


def positions(start: torch.Tensor, notch: torch.Tensor, n: int) -> torch.Tensor:
    """Window letters at each of n key presses (the rotors step before the current flows): (B, 3) → (B, n, 3)."""
    pos = torch.empty((start.shape[0], n, 3), dtype=torch.long, device=start.device)
    p = start
    for t in range(n):
        p = step(p, notch)
        pos[:, t] = p
    return pos


def all_positions(notch: torch.Tensor, n: int) -> torch.Tensor:
    """Window letters for all 26³ starts (row order of tensors.all_starts): (B, n, 3).

    The left rotor never kicks anything, so the stepping is computed for the 26² (mid, right) starts only and the
    left start is added afterwards: 26 times less work than stepping every start.
    """
    starts = tensors.all_starts(notch.device)  # (B, 3)
    pos = positions(starts[: 26 * 26], notch, n).repeat(26, 1, 1)  # (B, n, 3), left start 0 everywhere
    pos[..., 0] = (pos[..., 0] + starts[:, None, 0]) % 26
    return pos


def run_machine(text: torch.Tensor, m: dict, start: torch.Tensor, rings: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    """Encrypt (or, the same thing, decrypt) one text with B start positions at once: (T,), (B, 3) → (B, T)."""
    return encipher(text.expand(start.shape[0], -1), (positions(start, m["notch"], len(text)) - rings) % 26, m, s)


def keypress_matrices(m: dict, start: torch.Tensor, rings: torch.Tensor, s: torch.Tensor, n: int) -> torch.Tensor:
    """The 26×26 matrix E_t of each of the first n key presses for one setting: start (3,) → (n, 26, 26)."""
    offsets = (positions(start[None], m["notch"], n)[0] - rings) % 26  # (n, 3)
    r = [
        rotor_matrix(tensors.matrix(m["w"][k]), tensors.shift(-offsets[:, k]), tensors.shift(offsets[:, k]))
        for k in range(3)
    ]
    return keypress_matrix(tensors.matrix(s), r[2], r[1], r[0], tensors.matrix(m["u"]))  # R₁ = right rotor


def score(y: torch.Tensor, method: str, log_p: torch.Tensor) -> torch.Tensor:
    """How English-like every candidate decryption looks: (B, T) → (B,)."""
    c = letter_counts(y)
    return index_of_coincidence(c) if method == "ioc" else unigram_score(c, log_p)


def break_enigma(cipher: torch.Tensor, orders: list, rings: torch.Tensor, method: str) -> dict:
    """Score all 26³ start positions for every rotor order (empty plugboard, known rings).

    Returns scores (O, 26, 26, 26) indexed [order, left, mid, right] and the best order and start.
    """
    dev = cipher.device
    starts = tensors.all_starts(dev)  # (B, 3), B = 17,576
    s, log_p = torch.arange(26, device=dev), tensors.english_log_probs(dev)
    text = cipher.expand(len(starts), -1)  # (B, T): the same ciphertext for every candidate
    scores = torch.empty(len(orders), len(starts), device=dev)
    for i in steps(len(orders), "rotor orders"):
        m = tensors.machine(orders[i], dev)
        offsets = (all_positions(m["notch"], len(cipher)) - rings) % 26  # (B, T, 3)
        scores[i] = score(encipher(text, offsets, m, s), method, log_p)  # decrypt all B candidates, then score
    best = int(scores.argmax())
    order_i, start_i = divmod(best, len(starts))
    return {"scores": scores.reshape(len(orders), 26, 26, 26), "order": order_i, "start": starts[start_i]}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/enigma_solver/plot.py
"""Animate the search (score heatmap per rotor order, best score per order) and then the decrypted text appearing."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from . import tensors

LINE = 76  # letters per text line
TICKS = [0, 5, 10, 15, 20, 25]


def animate(heat, best, orders, previews, cipher, plain, best_i, best_start, method, path: Path) -> Path:
    """heat (O, 26, 26) best score over the left start for every (mid, right) start; best (O,) per order."""
    heat, best = heat.cpu().numpy(), best.cpu().numpy()
    n_orders, n_reveal = len(orders), 30
    every = math.ceil(n_orders / 30)  # at most 30 search frames: with 60 orders, 2 orders per frame
    events = [("search", i) for i in [*range(every - 1, n_orders - 1, every), n_orders - 1]]
    events += [("text", k) for k in range(1, n_reveal + 1)] + [("text", n_reveal)] * 6  # then hold the final text
    lo, hi = np.percentile(heat, 50), heat.max()

    fig = plt.figure(figsize=(9, 5.2))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.5, 1], width_ratios=[1, 1.5], hspace=0.45, wspace=0.3)
    ax_h, ax_b, ax_t = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, :])

    image = ax_h.imshow(heat[0], cmap=plt.get_cmap("viridis", 10), vmin=lo, vmax=hi, origin="lower")
    ax_h.set_xticks(TICKS, [tensors.ALPHABET[i] for i in TICKS])
    ax_h.set_yticks(TICKS, [tensors.ALPHABET[i] for i in TICKS])
    ax_h.set_xlabel("right rotor start")
    ax_h.set_ylabel("middle rotor start")
    ax_h.grid(False)
    (marker,) = ax_h.plot([], [], "o", ms=18, mfc="none", mec="#f59e0b", mew=2)
    title_h = ax_h.set_title("", fontsize=10)

    x = np.arange(n_orders)
    bars = ax_b.bar(x, np.zeros(n_orders), color="#9ca3af", width=0.8)
    if method == "ioc":
        ref = {"random 1/26": 1 / 26, "English": 0.066}
        ax_b.set_ylabel("index of coincidence")
    else:
        p = np.array(tensors.ENGLISH_FREQ) / sum(tensors.ENGLISH_FREQ)
        ref = {"random": np.log(p).mean(), "English": (p * np.log(p)).sum()}
        ax_b.set_ylabel("log-likelihood / letter")
    for (label, y), color in zip(ref.items(), ["#9ca3af", "#16a34a"]):
        ax_b.axhline(y, color=color, ls="--", lw=1)
        ax_b.text(n_orders - 0.5, y, label, ha="right", va="bottom", fontsize=8, color=color)
    span = max(best.max(), max(ref.values())) - min(best.min(), min(ref.values()))
    ax_b.set_ylim(min(best.min(), min(ref.values())) - 0.15 * span, max(best.max(), max(ref.values())) + 0.15 * span)
    ax_b.set_xlim(-0.8, n_orders - 0.2)
    ax_b.set_axisbelow(True)
    if n_orders <= 12:
        ax_b.set_xticks(x, ["-".join(o) for o in orders], fontsize=8)
    else:
        ax_b.set_xticks([])
        ax_b.set_xlabel(f"rotor order ({n_orders} orders)")
    title_b = ax_b.set_title("best score over all 26³ starts", fontsize=10)
    best_label = ax_b.text(best_i, 0, "", ha="center", va="bottom", fontsize=8, color="#1e40af")

    ax_t.axis("off")
    text = ax_t.text(0, 1, "", family="monospace", fontsize=9.5, va="top", transform=ax_t.transAxes)
    wrap = lambda s: "\n".join(s[i : i + LINE] for i in range(0, len(s), LINE))  # noqa: E731
    cipher_head = wrap(cipher[: 2 * LINE])

    def update(f: int):
        kind, k = events[f]
        done = k + 1 if kind == "search" else n_orders  # orders scored so far
        shown = k if kind == "search" else best_i
        leader = int(np.argmax(best[:done]))
        image.set_data(heat[shown])
        title_h.set_text(f"{'-'.join(orders[shown])}: best over left start")
        if shown == best_i and done > best_i:
            marker.set_data([best_start[2].item()], [best_start[1].item()])
        else:
            marker.set_data([], [])
        for i, bar in enumerate(bars):
            bar.set_height(best[i] if i < done else 0)
            bar.set_color("#2563eb" if i == leader else ("#f59e0b" if i == shown and kind == "search" else "#9ca3af"))
        if done > best_i:
            best_label.set_position((best_i, best[best_i]))
            best_label.set_text("-".join(orders[best_i]))
        title_b.set_text(f"best score over all 26³ starts — {done}/{n_orders} orders")
        if kind == "search":
            text.set_text(f"ciphertext\n{cipher_head}\n\nbest so far: {previews[leader]}")
        else:
            n = math.ceil(len(plain) * k / n_reveal)
            key = f"{'-'.join(orders[best_i])} at {tensors.to_text(torch.as_tensor(best_start))}"
            text.set_text(f"decrypted with rotors {key}\n{wrap(plain[:n])}")

    fig.subplots_adjust(left=0.07, right=0.98, top=0.93, bottom=0.03)
    return save_gif(fig, update, frame_indices(len(events), 120), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'message': 'weather',
    'rotor_order': 'IV-II-V',
    'start_left': 'T',
    'start_middle': 'D',
    'start_right': 'X',
    'n_letters': 200,
    'score': 'ioc',
    'search_rotors': 'I-V'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.enigma_solver")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.enigma_solver.main")
problem_dir = Path("problems/others/enigma_solver")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
